# 04 — End-to-End Code Generation Pipeline Demo

Demonstrates the complete HaluGuard pipeline on individual and batched RepoBench examples.

**Pipeline stages:**
```
RepoBench example
  ↓
  CodeBERT embedding (query + chunks)
  ↓
  HCCS scoring (best trained model)
  ↓
  Type-router pre-emptive boost
  ↓
  DeepSeek-Coder generation (top-5 chunks)
  ↓
  Sandbox execution (execute_code)
  ↓  [if failed]
  Error → type_router boost → re-rank → retry (EFL, max 3x)
  ↓
  Metrics: Exact Match, Edit Similarity
```

**Requires:**
- Notebook 01: embeddings computed
- Notebook 02: best checkpoint saved to `checkpoints/`
- Notebook 03 (optional): for comparison with baselines

## 0. Setup

In [ ]:
import subprocess, sys, os
try:
    from google.colab import drive
    drive.mount('/content/drive')
    REPO_DIR = '/content/drive/MyDrive/HaluGuard'
    sys.path.insert(0, REPO_DIR)
    os.chdir(REPO_DIR)
    DRIVE_ROOT = REPO_DIR
except ImportError:
    DRIVE_ROOT = None
    REPO_DIR = os.path.abspath('..')

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[dev]"], check=True)

In [ ]:
import json, time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import numpy as np
import torch
from transformers import AutoModel, AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

from haluguard.models import MODEL_REGISTRY, build_model
from haluguard.hccs import HCCSScorer, embed_code, batch_embed
from haluguard.efl import run_efl, execute_code, build_completion_prompt as efl_prompt
from haluguard.generate import generate_next_line, build_completion_prompt
from haluguard.type_router import predict_boost, boost_scores
from haluguard.baselines import cosine_scores, bm25_select, no_context_select
from haluguard.evaluate import exact_match, edit_similarity, compute_metrics, compute_retrieval_summary

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')
if DEVICE == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Load Models

In [ ]:
BASE_DIR    = Path(REPO_DIR)
EMB_DIR     = BASE_DIR / 'data' / 'embeddings'
CKPT_DIR    = Path(DRIVE_ROOT) / 'checkpoints' if DRIVE_ROOT else BASE_DIR / 'checkpoints'
RESULTS_DIR = BASE_DIR / 'data' / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# ── Load CodeBERT encoder ────────────────────────────────────────────────────
ENCODER_NAME = 'microsoft/codebert-base'
cb_tokenizer = AutoTokenizer.from_pretrained(ENCODER_NAME)
cb_encoder   = AutoModel.from_pretrained(ENCODER_NAME).to(DEVICE)
cb_encoder.eval()
print('CodeBERT loaded')

# ── Load best HCCS scorer ────────────────────────────────────────────────────
# Read metadata to find the best checkpoint
meta_path = CKPT_DIR / 'best_model_meta.json'
if meta_path.exists():
    with open(meta_path) as f:
        meta = json.load(f)
    best_model_name = meta['best_model']
    print(f'Best model from metadata: {best_model_name}  (test MRR={meta.get("test_mrr", "?"):.4f})')
else:
    best_model_name = 'interaction_mlp'   # fallback
    print(f'No metadata found — using fallback: {best_model_name}')

# Try to load the new-style model checkpoint
ckpt_path = CKPT_DIR / f'{best_model_name}_best.pt'
if ckpt_path.exists() and best_model_name in MODEL_REGISTRY:
    from haluguard.models import InteractionMLP, DualEncoder, BilinearScorer
    # Instantiate correct class and load weights
    hccs_scorer = build_model(best_model_name)
    hccs_scorer.load_state_dict(torch.load(ckpt_path, map_location='cpu'))
    hccs_scorer = hccs_scorer.to(DEVICE).eval()
    print(f'Loaded {best_model_name} scorer from {ckpt_path.name}')
else:
    # Fall back to legacy HCCSScorer
    legacy_ckpt = next((CKPT_DIR / f for f in ['hccs_unixcoder_last3_best.pt',
                                                 'hccs_codebert_full_best.pt']
                         if (CKPT_DIR / f).exists()), None)
    if legacy_ckpt:
        hccs_scorer = HCCSScorer.load(legacy_ckpt).to(DEVICE).eval()
        print(f'Loaded legacy HCCSScorer from {legacy_ckpt.name}')
    else:
        raise FileNotFoundError('No HCCS checkpoint found. Run notebook 02 first.')

In [ ]:
# ── Load DeepSeek-Coder ──────────────────────────────────────────────────────
GEN_MODEL_NAME = 'deepseek-ai/deepseek-coder-1.3b-base'
gen_tokenizer = AutoTokenizer.from_pretrained(GEN_MODEL_NAME)
gen_model = AutoModelForCausalLM.from_pretrained(
    GEN_MODEL_NAME, dtype=torch.float16, device_map='auto'
)
gen_model.eval()
print(f'DeepSeek-Coder loaded on {next(gen_model.parameters()).device}')

def gen_fn(prompt: str) -> str:
    """Generate a single next line."""
    return generate_next_line(
        prompt, gen_tokenizer, gen_model,
        device=DEVICE, max_new_tokens=64, temperature=0.2,
    )

## 2. Load Pre-computed Embeddings + Dataset

In [ ]:
from datasets import load_dataset

ds = load_dataset('tianyang/repobench_python_v1.1', split='cross_file_first')
print(f'RepoBench: {len(ds)} examples')

# Load pre-computed embeddings (same as used in training)
def _try_paths(*paths):
    for p in paths:
        if Path(p).exists():
            return Path(p)
    raise FileNotFoundError(f'None found: {paths}')

query_embs_path = _try_paths(
    EMB_DIR / 'query_embeddings__codebert__last3.pt',
    EMB_DIR / 'query_embeddings.pt',
)
chunk_embs_path = _try_paths(
    EMB_DIR / 'chunk_embeddings__codebert.pt',
    EMB_DIR / 'chunk_embeddings.pt',
)

query_embs   = torch.load(query_embs_path, map_location='cpu').float()
chunk_embs   = torch.load(chunk_embs_path, map_location='cpu')
gold_indices = torch.load(EMB_DIR / 'gold_indices.pt', map_location='cpu')
test_indices = torch.load(EMB_DIR / 'test_indices.pt', map_location='cpu') \
               if (EMB_DIR / 'test_indices.pt').exists() else list(range(100))

if isinstance(gold_indices, torch.Tensor):
    gold_indices = gold_indices.tolist()

print(f'Query embs : {query_embs.shape}')
print(f'Test examples: {len(test_indices)}')

## 3. Single Example — Step-by-Step Walkthrough

In [ ]:
# Pick one test example to walk through in detail
DEMO_IDX = test_indices[0]
ex = ds[DEMO_IDX]

print(f'Example index: {DEMO_IDX}')
print(f'Gold snippet index: {ex["gold_snippet_index"]}')
print(f'Number of context chunks: {len(ex["context"])}')
print(f'Ground truth next line: {ex["next_line"]!r}')
print(f'\nLast 3 lines of cropped_code:')
for line in ex['cropped_code'].splitlines()[-3:]:
    print(f'  {line}')

In [ ]:
# ── Step 1: Embed query + chunks ─────────────────────────────────────────────
q_emb  = query_embs[DEMO_IDX].numpy()     # pre-computed
c_embs = chunk_embs[DEMO_IDX].numpy()     # pre-computed
n_chunks = c_embs.shape[0]
print(f'Query embedding: {q_emb.shape}')
print(f'Chunk embeddings: {c_embs.shape}  ({n_chunks} chunks)')

# ── Step 2: HCCS scoring ─────────────────────────────────────────────────────
if hasattr(hccs_scorer, 'score'):
    # New-style model (haluguard.models)
    q_t = torch.tensor(q_emb, dtype=torch.float32).to(DEVICE)
    c_t = torch.tensor(c_embs, dtype=torch.float32).to(DEVICE)
    with torch.no_grad():
        hccs_raw = hccs_scorer.score(q_t, c_t).cpu().numpy()
else:
    # Legacy HCCSScorer
    hccs_raw = hccs_scorer.score_chunks(q_emb, c_embs, device=DEVICE)

# ── Step 3: Type-router pre-emptive boost ────────────────────────────────────
boosts = predict_boost(ex['cropped_code'])
hccs_boosted = boost_scores(hccs_raw, ex['context'], boosts)

# ── Step 4: Select top-5 chunks ──────────────────────────────────────────────
TOP_K = 5
top_indices_hccs   = list(np.argsort(hccs_boosted)[::-1][:TOP_K])
top_indices_cosine = list(np.argsort(cosine_scores(q_emb, c_embs))[::-1][:TOP_K])
gold_index = int(ex['gold_snippet_index'])

print(f'\nGold chunk index : {gold_index}')
print(f'HCCS top-5       : {top_indices_hccs}  (gold in top-5: {gold_index in top_indices_hccs})')
print(f'Cosine top-5     : {top_indices_cosine}  (gold in top-5: {gold_index in top_indices_cosine})')
print(f'\nDetected boosts  : {boosts}')

In [ ]:
# ── Step 5: Generate next line (HCCS context) ────────────────────────────────
snippets_hccs = [ex['context'][j]['snippet'] for j in top_indices_hccs]
prompt_hccs   = build_completion_prompt(ex['cropped_code'], ex['import_statement'], snippets_hccs)

print(f'Prompt length: {len(prompt_hccs)} chars')
pred_hccs = gen_fn(prompt_hccs)

print(f'\n── Generation Result ──')
print(f'Predicted : {pred_hccs!r}')
print(f'Ground truth: {ex["next_line"]!r}')
print(f'EM: {exact_match(pred_hccs, ex["next_line"]):.0f}')
print(f'ES: {edit_similarity(pred_hccs, ex["next_line"]):.3f}')

In [ ]:
# ── Step 6: Execution Feedback Loop (EFL) ───────────────────────────────────
# Run EFL — generate, execute in sandbox, retry on failure with error-guided re-ranking

efl_result = run_efl(
    cropped_code=ex['cropped_code'],
    import_statement=ex['import_statement'],
    contexts=ex['context'],
    scores=hccs_boosted,
    generate_fn=gen_fn,
    top_k=TOP_K,
    max_iterations=3,
    timeout=10,
)

print('── EFL Result ──')
print(f'Best prediction: {efl_result.code!r}')
print(f'Passed sandbox : {efl_result.passed}')
print(f'EFL iterations : {efl_result.iterations}')
print(f'EM (EFL)       : {exact_match(efl_result.code, ex["next_line"]):.0f}')
print(f'ES (EFL)       : {edit_similarity(efl_result.code, ex["next_line"]):.3f}')

if len(efl_result.history) > 1:
    print('\nEFL history:')
    for i, hr in enumerate(efl_result.history):
        print(f'  Iter {i}: passed={hr.passed}  error={hr.error_type}')

## 4. Code-Generation Benchmark — All 7 Architectures + Baselines (RepoBench-P)

Runs the full HaluGuard pipeline on `N_EVAL = 150` held-out RepoBench test examples
for **every** scorer architecture in `MODEL_REGISTRY`, plus four reference baselines:

| Group | Method |
|-------|--------|
| Lower bound | `no_context` |
| Lexical | `bm25` |
| Semantic | `cosine` (CodeBERT) |
| Oracle | `gold_only` |
| **HaluGuard** | `dual_encoder`, `dual_encoder_deep`, `listwise_mlp`, `pairwise_mlp`, `interaction_mlp`, `bilinear`, `ensemble` |
| + EFL | Every HaluGuard arch, with the Execution Feedback Loop enabled |

**Metrics:** Exact Match (EM), Edit Similarity (ES), CodeBLEU.

**Runtime:** ≈ 25–40 min on a T4 GPU for `N_EVAL = 150`.  Increase `N_EVAL` for
full-test eval (~9 h).  DeepSeek-Coder is loaded once and shared across methods.

Output:
- `data/results/repobench_p_table.json` — one row per method
- `data/results/repobench_p_details.jsonl` — every (method, idx, pred, gt) tuple

In [ ]:
# ── RepoBench-P benchmark: all 7 architectures + 4 baselines ─────────────
# DeepSeek-Coder is reused across methods — we load the model and scorers once.
from haluguard.baselines import (
    bm25_select, cosine_scores, full_context_select,
    gold_only_select, no_context_select,
)
from haluguard.models import EnsembleScorer
from haluguard.evaluate import compute_metrics

N_EVAL = 150   # tune between 100 and 200; 1200+ ≈ 9 h on T4
TOP_K = 5
EFL_ITERATIONS = 3
EFL_TIMEOUT = 10

batch_indices = list(test_indices[:N_EVAL])
print(f'[repobench-p] evaluating {len(batch_indices)} test examples')


# -----------------------------------------------------------------------------
# Load every available trained architecture ONCE.
# -----------------------------------------------------------------------------
loaded_scorers: Dict[str, torch.nn.Module] = {}
for arch_name in MODEL_REGISTRY:
    if arch_name == 'ensemble':
        continue
    ckpt = CKPT_DIR / f'{arch_name}_best.pt'
    if not ckpt.exists():
        print(f'[repobench-p] skipping {arch_name} — missing {ckpt.name}')
        continue
    try:
        model = build_model(arch_name)
        model.load_state_dict(torch.load(ckpt, map_location='cpu'), strict=False)
        loaded_scorers[arch_name] = model.to(DEVICE).eval()
        print(f'[repobench-p] loaded {arch_name} from {ckpt.name}')
    except Exception as err:
        print(f'[repobench-p] failed to load {arch_name}: {err}')

# Ensemble — requires top-3 member checkpoints from notebook 02 metadata.
meta_path = CKPT_DIR / 'best_model_meta.json'
ens_ckpt  = CKPT_DIR / 'ensemble_best.pt'
if meta_path.exists() and ens_ckpt.exists():
    meta_info = json.loads(meta_path.read_text())
    top3 = list(meta_info.get('top3_ensemble', []))
    members = [loaded_scorers[n] for n in top3 if n in loaded_scorers]
    if len(members) >= 2:
        try:
            ensemble = EnsembleScorer.load(ens_ckpt, scorers=members).to(DEVICE).eval()
            loaded_scorers['ensemble'] = ensemble
            print(f'[repobench-p] loaded ensemble (members={top3})')
        except Exception as err:
            print(f'[repobench-p] failed to load ensemble: {err}')
    else:
        print(f'[repobench-p] ensemble needs ≥2 members (found {len(members)}) — skipped')
else:
    print('[repobench-p] ensemble checkpoint or metadata missing — skipped')


# -----------------------------------------------------------------------------
# Helpers
# -----------------------------------------------------------------------------
def _score_arch(model: torch.nn.Module, q_emb: np.ndarray, c_embs: np.ndarray) -> np.ndarray:
    qt = torch.as_tensor(q_emb, dtype=torch.float32, device=DEVICE)
    ct = torch.as_tensor(c_embs, dtype=torch.float32, device=DEVICE)
    with torch.no_grad():
        logits = model.score(qt, ct)
    return logits.detach().cpu().numpy()


results_by_method: Dict[str, List[Tuple[str, str]]] = {}
details_path = RESULTS_DIR / 'repobench_p_details.jsonl'
details_path.parent.mkdir(parents=True, exist_ok=True)
details_path.write_text('')   # fresh file — overwrite any previous run


def _save_pair(method: str, idx: int, pred: str, gt: str) -> None:
    with details_path.open('a') as f:
        f.write(json.dumps({
            'method': method, 'idx': int(idx), 'pred': pred, 'gt': gt,
        }) + '\n')


def _run_method(name: str, select_from_ex) -> None:
    """Generic runner — `select_from_ex(idx, ex) -> List[int]` chooses chunk indices."""
    pairs: List[Tuple[str, str]] = []
    print(f'\n[{name}] starting {len(batch_indices)} examples')
    t0 = time.time()
    for idx in tqdm(batch_indices, desc=name):
        ex = ds[idx]
        selected = select_from_ex(idx, ex)
        snippets = [ex['context'][j]['snippet'] for j in selected]
        prompt = build_completion_prompt(
            ex['cropped_code'], ex['import_statement'], snippets,
        )
        pred = gen_fn(prompt)
        pairs.append((pred, ex['next_line']))
        _save_pair(name, idx, pred, ex['next_line'])
    results_by_method[name] = pairs
    m = compute_metrics([p for p, _ in pairs], [g for _, g in pairs])
    print(f'[{name}] EM={m["em"]:.3f}  ES={m["es"]:.3f}  CodeBLEU={m["codebleu"]:.3f} '
          f'(elapsed {time.time()-t0:.1f}s)')


def _run_halu(arch_name: str, model: torch.nn.Module, *, use_efl: bool) -> None:
    """Run the HaluGuard pipeline for one scorer, with or without EFL."""
    method_name = f'{arch_name}_efl' if use_efl else arch_name
    pairs: List[Tuple[str, str]] = []
    print(f'\n[{method_name}] starting {len(batch_indices)} examples')
    t0 = time.time()
    for idx in tqdm(batch_indices, desc=method_name):
        ex = ds[idx]
        q_emb = query_embs[idx].numpy()
        c_embs = chunk_embs[idx].numpy()
        if c_embs.shape[0] == 0:
            pairs.append(('', ex['next_line']))
            _save_pair(method_name, idx, '', ex['next_line'])
            continue

        raw = _score_arch(model, q_emb, c_embs)
        boosted = boost_scores(raw, ex['context'], predict_boost(ex['cropped_code']))

        if use_efl:
            efl = run_efl(
                cropped_code=ex['cropped_code'],
                import_statement=ex['import_statement'],
                contexts=ex['context'],
                scores=boosted,
                generate_fn=gen_fn,
                top_k=TOP_K,
                max_iterations=EFL_ITERATIONS,
                timeout=EFL_TIMEOUT,
                verbose=False,
            )
            pred = efl.code
        else:
            top_idx = list(np.argsort(boosted)[::-1][:TOP_K])
            snippets = [ex['context'][j]['snippet'] for j in top_idx]
            prompt = build_completion_prompt(
                ex['cropped_code'], ex['import_statement'], snippets,
            )
            pred = gen_fn(prompt)

        pairs.append((pred, ex['next_line']))
        _save_pair(method_name, idx, pred, ex['next_line'])

    results_by_method[method_name] = pairs
    m = compute_metrics([p for p, _ in pairs], [g for _, g in pairs])
    print(f'[{method_name}] EM={m["em"]:.3f}  ES={m["es"]:.3f}  CodeBLEU={m["codebleu"]:.3f} '
          f'(elapsed {time.time()-t0:.1f}s)')


# ── Baselines ────────────────────────────────────────────────────────────
_run_method('no_context', lambda idx, ex: no_context_select())
_run_method('bm25',       lambda idx, ex: bm25_select(ex['cropped_code'], ex['context'], top_k=TOP_K))
_run_method(
    'cosine',
    lambda idx, ex: list(
        np.argsort(cosine_scores(query_embs[idx].numpy(), chunk_embs[idx].numpy()))[::-1][:TOP_K]
    ),
)
_run_method('gold_only', lambda idx, ex: gold_only_select(ex['gold_snippet_index']))


# ── HaluGuard: all 7 architectures, first without EFL then with EFL ───────
for arch_name, model in loaded_scorers.items():
    _run_halu(arch_name, model, use_efl=False)

for arch_name, model in loaded_scorers.items():
    _run_halu(arch_name, model, use_efl=True)

print(f'\n[repobench-p] done — evaluated {len(results_by_method)} methods')

In [ ]:
# ── Aggregate + print RepoBench-P table ───────────────────────────────────
from haluguard.evaluate import compute_metrics_table

repo_p_table = compute_metrics_table(results_by_method)

print(f'\n── RepoBench-P Code-Generation Results (n={N_EVAL}) ──')
header = f'{"method":<26} {"EM":>7} {"ES":>7} {"CodeBLEU":>9}'
print(header)
print('-' * len(header))
for row in repo_p_table:
    print(f'{row["method"]:<26} {row["em"]:>7.3f} {row["es"]:>7.3f} {row["codebleu"]:>9.3f}')

# Persist
table_path = RESULTS_DIR / 'repobench_p_table.json'
with table_path.open('w') as f:
    json.dump(repo_p_table, f, indent=2)
print(f'\nSaved table → {table_path.name}')
print(f'Saved per-example details → {details_path.name}')

## 5. Error Analysis — Where Does HCCS Help vs Hurt?

In [ ]:
# ── Error analysis: pick the best HaluGuard arch vs cosine baseline ──────
# Shows concrete cases where HaluGuard's best architecture beats cosine.
best_row = max(
    (r for r in repo_p_table
     if r['method'] in loaded_scorers and not r['method'].endswith('_efl')),
    key=lambda r: r['em'],
    default=None,
)
if best_row is None:
    print('No HaluGuard arch results found — skipping error analysis.')
else:
    best_name = best_row['method']
    print(f'Best HaluGuard arch on EM: {best_name} (EM={best_row["em"]:.3f})')

    halu_pairs   = results_by_method[best_name]
    cosine_pairs = results_by_method.get('cosine', [])

    # `hccs_wins` / `cosine_wins` — hallucination-context-scoring wins over
    # the cosine baseline on exact-match.  Naming preserved for tests.
    hccs_wins, cosine_wins, ties = [], [], []
    for step, idx in enumerate(batch_indices):
        ex = ds[idx]
        pred_halu   = halu_pairs[step][0]
        pred_cosine = cosine_pairs[step][0] if cosine_pairs else ''
        gt = ex['next_line']
        em_h = exact_match(pred_halu, gt)
        em_c = exact_match(pred_cosine, gt)
        if em_h > em_c:
            hccs_wins.append({'idx': idx, 'gt': gt, 'halu': pred_halu, 'cosine': pred_cosine})
        elif em_c > em_h:
            cosine_wins.append({'idx': idx, 'gt': gt, 'halu': pred_halu, 'cosine': pred_cosine})
        else:
            ties.append(idx)

    print(f'{best_name} wins  : {len(hccs_wins):>3}')
    print(f'cosine wins       : {len(cosine_wins):>3}')
    print(f'ties              : {len(ties):>3}')

    print(f'\n── First 3 cases where {best_name} beats cosine ──')
    for w in hccs_wins[:3]:
        print(f"  GT       : {w['gt']!r}")
        print(f"  HaluGuard: {w['halu']!r}")
        print(f"  Cosine   : {w['cosine']!r}")
        print()
